# Binary Search


## Topic overview

Halve the search space each step over a monotonic predicate.

## Pattern-recognition rules

- Search a sorted array.
- Search on the *answer* (parametric search).
- First/last occurrence via boundary shifting.

## Common data structures

- Sorted arrays
- Any monotonic predicate space

## Standard complexity expectations

- O(log n) per query.

## Common mistakes

- Off-by-one on `lo`/`hi` when converging.
- Overflow in `(lo+hi)//2` in non-Python languages.

## Original illustrative example

In [ ]:
# Replace with an ORIGINAL example. Do not paste external
# problem statements. See src/algorithms/ for reusable helpers.
example_input = []
example_expected = None

## Add solved problems below

Each new sub-section should follow the template in `../templates/notebook_template.ipynb`.

1. [Search a 2D Matrix](#search-a-2d-matrix)
2. [Reverse Nodes in k-Group](#reverse-nodes-in-k-group)
3. [Koko Eating Bananas](#koko-eating-bananas)
4. [Find Minimum in Rotated Sorted Array](#find-minimum-in-rotated-sorted-array)


# Search a 2D Matrix

## Metadata

- Source: NeetCode / LeetCode 74
- Problem URL: https://leetcode.com/problems/search-a-2d-matrix/
- Difficulty: Medium
- Topic: Binary Search
- Date started: 2026-09-01
- Date solved: 2026-09-01
- Current mastery level: 1
- Last reviewed: 2026-09-01
- Next review:

## Problem statement in my own words

You get an `m x n` integer grid and a target number. Decide whether that
target appears anywhere in the grid.

Two sorting facts make this a binary-search problem, not a scan:

1. Every row is sorted left to right (non-decreasing).
2. The first value of each row is **greater than** the last value of the
   previous row.

Those two facts mean the whole matrix is globally sorted if you read it
row by row. Mentally flatten

```
[
    [ 1,  2,  4,  8],
    [10, 11, 12, 13],
    [14, 20, 30, 40]
]
```

and you get the ordinary sorted array

```
[1, 2, 4, 8, 10, 11, 12, 13, 14, 20, 30, 40]
```

Return `True` if `target` is present, otherwise `False`. The intended
bound is $O(\log(mn))$ — one binary search over the virtual 1-D array,
**without** allocating a flattened copy.

## Inputs, outputs, and constraints

- Inputs: `matrix` (`m` rows, `n` columns of ints) and `target` (int)
- Outputs: `bool` — whether `target` exists in `matrix`
- Constraints:
  - $1 \le m, n \le 100$
  - $-10^4 \le$ `matrix[i][j]`, `target` $\le 10^4$
  - each row is sorted non-decreasing
  - `matrix[r][0] > matrix[r-1][n-1]` for every row `r >= 1`

## Examples

| Input | Expected | Notes |
|---|---|---|
| `matrix = [[1,2,4,8],[10,11,12,13],[14,20,30,40]]`, `target = 10` | `True` | target sits at `[1][0]` |
| `matrix = [[1,2,4,8],[10,11,12,13],[14,20,30,40]]`, `target = 15` | `False` | 15 falls in the gap between 14 and 20 |

Same $3 \times 4$ grid in both examples. Flattened view:

```
index:  0  1  2  3   4   5   6   7   8   9  10  11
value: [1, 2, 4, 8, 10, 11, 12, 13, 14, 20, 30, 40]
```

## Initial observations

- A nested scan is correct but $O(mn)$ — too slow for the asked bound.
- Binary-searching **each row** is $O(m \log n)$. Better, still not
  $O(\log(mn))$.
- Because the matrix is globally sorted, it behaves like one sorted
  array of length $N = mn$. Binary search on that virtual array is
  $O(\log(mn))$.
- The only extra work is mapping a 1-D index `mid` back to `(row, col)`
  so we never allocate the flattened copy.

## Brute-force approach

### Why it works

Walk every cell. If any equals `target`, return `True`. The matrix
guarantees are unused; this is just exhaustive search.

### Implementation

In [ ]:
from typing import List


def brute_force(matrix: List[List[int]], target: int) -> bool:
    for row in matrix:
        for value in row:
            if value == target:
                return True
    return False

### Complexity

- Time: $O(mn)$
- Space: $O(1)$

## Optimized insight

Treat the matrix as one long sorted array of length $mn$, then run the
same binary-search loop you use on a 1-D sorted list.

**Do not actually flatten.** Building `flat` is already $O(mn)$ and
destroys the required $O(\log(mn))$ bound. Convert the virtual index
instead:

$$
\text{row} = \left\lfloor \frac{\text{mid}}{n} \right\rfloor, \qquad
\text{col} = \text{mid} \bmod n
$$

In Python, with `n = cols`:

```python
row = mid // cols
col = mid % cols
```

Division says which row you have reached. Remainder says how far into
that row you have moved.

For $n = 4$:

```
Virtual index:

  0   1   2   3
  4   5   6   7
  8   9  10  11

Matrix value:

  1   2   4   8
 10  11  12  13
 14  20  30  40
```

`mid = 6` → `row = 6 // 4 = 1`, `col = 6 % 4 = 2` → `matrix[1][2] = 12`.

`mid = 10` → `row = 10 // 4 = 2`, `col = 10 % 4 = 2` → `matrix[2][2] = 30`.

## Optimized approach

1. Set `left = 0`, `right = m * n - 1`.
2. While `left <= right`:
   - `mid = (left + right) // 2`
   - read `value = matrix[mid // n][mid % n]`
   - equal → `True`
   - too small → drop the left half (`left = mid + 1`)
   - too large → drop the right half (`right = mid - 1`)
3. If the window empties, return `False`.

### Step-by-step trace

Target `10` on the $3 \times 4$ example. Search space starts as
`[0, 11]`.

| Step | left | right | mid | (row, col) | value | Note |
|---|---|---|---|---|---|---|
| 1 | 0 | 11 | 5 | (1, 1) | 11 | 11 > 10 → `right = 4` |
| 2 | 0 | 4 | 2 | (0, 2) | 4 | 4 < 10 → `left = 3` |
| 3 | 3 | 4 | 3 | (0, 3) | 8 | 8 < 10 → `left = 4` |
| 4 | 4 | 4 | 4 | (1, 0) | 10 | found → `True` |

Flattened view of the same process:

```
[1, 2, 4, 8, 10, 11, 12, 13, 14, 20, 30, 40]
                ↑ mid=5 (11)  → drop right
[1, 2, 4, 8, 10]
       ↑ mid=2 (4)            → drop left
[8, 10]
 ↑ mid=3 (8)                  → drop left
[10]
  ↑ found
```

For `target = 15`, binary search closes the window on the gap
`14 < 15 < 20`. Then `left > right` and the answer is `False`.

In [ ]:
from typing import List


class Solution:
    def searchMatrix(self, matrix: List[List[int]], target: int) -> bool:
        rows = len(matrix)
        cols = len(matrix[0])

        left = 0
        right = rows * cols - 1

        while left <= right:
            mid = (left + right) // 2

            row = mid // cols
            col = mid % cols
            value = matrix[row][col]

            if value == target:
                return True
            if value < target:
                left = mid + 1
            else:
                right = mid - 1

        return False


def optimized(matrix: List[List[int]], target: int) -> bool:
    return Solution().searchMatrix(matrix, target)

## Complexity analysis

Let $N = mn$ be the number of cells.

Each comparison halves the remaining window:

$$
N \rightarrow N/2 \rightarrow N/4 \rightarrow \cdots \rightarrow 1
$$

so the number of iterations is $\log_2 N = \log(mn)$.

- Time: $O(\log(mn))$
- Space: $O(1)$ — only `left`, `right`, `mid`, `row`, `col`, `value`

## Edge cases

- Single cell: `[[7]]` — window is `[0, 0]`; one comparison decides it
- Target smaller than `matrix[0][0]` — search walks left and empties
- Target larger than `matrix[-1][-1]` — search walks right and empties
- Target equals a row boundary (`8` or `10` in the example) — the
  `mid // cols` / `mid % cols` map still lands on the correct cell
- Value in the gap between two rows (`15` between 14 and 20) — `False`
- Negative numbers are legal under the constraints; the same loop works
  because order, not sign, is what binary search needs

## Testing

In [ ]:
matrix = [
    [1, 2, 4, 8],
    [10, 11, 12, 13],
    [14, 20, 30, 40],
]

assert optimized(matrix, 10) is True
assert optimized(matrix, 15) is False
assert optimized(matrix, 1) is True
assert optimized(matrix, 40) is True
assert optimized(matrix, 8) is True
assert optimized(matrix, 14) is True
assert optimized(matrix, 0) is False
assert optimized(matrix, 41) is False
assert optimized([[7]], 7) is True
assert optimized([[7]], 3) is False
assert brute_force(matrix, 10) is True
assert brute_force(matrix, 15) is False

print("Search a 2D Matrix checks passed")

## Alternative approaches

- Scan every cell: $O(mn)$ time, $O(1)$ space. Correct, too slow.
- Binary search each row: $O(m \log n)$. Uses the per-row sort but
  ignores the global order between rows.
- Binary search the first column to pick a row, then binary search that
  row: $O(\log m + \log n) = O(\log(mn))$. Same bound, two searches.
  The virtual-index version is one loop and the same idea.
- Materialize `flat` then binary search it: still $O(\log(mn))$
  comparisons, but $O(mn)$ time and space to build `flat`. Do not do this.

## Mistakes I made

- Forgetting that actually flattening the matrix is already linear.
- Using `mid % rows` instead of `mid % cols` when mapping the index.
- Exclusive right bound (`right = mn`) with a `left <= right` loop —
  pick one convention and stick to it. Here: inclusive `[left, right]`.

## Pattern recognition

This is 1-D binary search plus an index encoding. Whenever a 2-D
structure is **row-major and globally sorted**, a flat index

```text
index = row * cols + col
```

and its inverse

```text
row = index // cols
col = index % cols
```

let you reuse the 1-D algorithm.

## Related problems

- Binary Search (1-D) — same loop, no `(row, col)` map
- Search a 2D Matrix II — rows and columns sorted, but **not** globally
  sorted across row boundaries; the flatten trick does not apply
- Find First and Last Position of Element in Sorted Array — same search
  space idea, different boundary policy
- Koko Eating Bananas / capacity problems — binary search on the
  *answer*, not on array indices

## Real-world or engineering connection

Row-major layouts (images, matrices, C arrays) already store a 2-D grid
as one contiguous buffer. The `index // cols`, `index % cols` map is
exactly how those buffers are addressed. Here we use that addressing so
we can binary-search the buffer without copying it.

## Final takeaways

The two lines that carry the whole problem:

```python
row = mid // cols
col = mid % cols
```

Same binary-search skeleton as the 1-D problem. The matrix is only a
view over a sorted array of length $mn$.

## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-09-01 | 1 | First write-up: virtual flatten + binary search |

# Reverse Nodes in k-Group

## Metadata

- Source: NeetCode / LeetCode 25
- Problem URL: https://leetcode.com/problems/reverse-nodes-in-k-group/
- Difficulty: Hard
- Topic: Linked Lists (pointer reversal)
- Date started: 2026-09-01
- Date solved: 2026-09-01
- Current mastery level: 1
- Last reviewed: 2026-09-01
- Next review:

When $k = 2$, this is Swap Nodes in Pairs. The extra work is grouping,
stopping early when a leftover suffix is shorter than $k$, and
reconnecting each reversed block without losing pointers.

## Problem statement in my own words

You are given the head of a singly linked list and a positive integer
$k$. Reverse the list **in groups of exactly $k$ nodes**, then return
the new head.

Rules:

- Only complete groups of $k$ are reversed.
- If the leftover suffix has fewer than $k$ nodes, leave that suffix
  in its original order.
- Rewire the nodes. Do not swap values inside nodes.

Example with $k = 2$:

```
1 → 2 → 3 → 4 → 5
[1 → 2] [3 → 4] [5]
[2 → 1] [4 → 3] [5]
2 → 1 → 4 → 3 → 5
```

Example with $k = 3$:

```
1 → 2 → 3 → 4 → 5
[1 → 2 → 3] [4 → 5]
[3 → 2 → 1] [4 → 5]
3 → 2 → 1 → 4 → 5
```

## Inputs, outputs, and constraints

- Inputs: `head` of a singly linked list, and `k`
- Outputs: head of the modified list
- Constraints:
  - $n$ = number of nodes
  - $1 \le k \le n \le 5000$
  - $0 \le$ `Node.val` $\le 1000$
  - $k$ is always at most the length of the list

## Examples

| Input | Expected | Notes |
|---|---|---|
| `head = [1,2,3,4,5]`, `k = 2` | `[2,1,4,3,5]` | two complete pairs; leftover `5` stays |
| `head = [1,2,3,4,5]`, `k = 3` | `[3,2,1,4,5]` | one complete triple; leftover `4 → 5` stays |

## Initial observations

- Reversing a linked list is the easy part. The hard part is reversing
  **exactly $k$ nodes**, hooking that block back to the previous group
  and the unused suffix, then repeating.
- The first group's new head becomes the answer head. A dummy node
  sitting before `head` makes the first group look like every later
  group, so you do not special-case it.
- Walking $k$ steps from the node before a group tells you whether a
  full group exists. If that walk falls off the list, stop.
- The one trick in the reversal loop: initialize `prev` to the node
  *after* the group, not `None`. The new tail then already points at
  the remainder.

## Brute-force approach

### Why it works

Collect every node into an array, reverse each complete chunk of $k$
nodes, then rewire `next` pointers from the new order. Correct, but it
uses $O(n)$ extra space.

### Implementation

In [ ]:
from typing import List, Optional


class ListNode:
    def __init__(self, val: int = 0, next: Optional["ListNode"] = None):
        self.val = val
        self.next = next


def to_list(head: Optional[ListNode]) -> List[int]:
    values: List[int] = []
    curr = head
    while curr is not None:
        values.append(curr.val)
        curr = curr.next
    return values


def from_list(values: List[int]) -> Optional[ListNode]:
    dummy = ListNode(0)
    curr = dummy
    for value in values:
        curr.next = ListNode(value)
        curr = curr.next
    return dummy.next


def brute_force_k_group(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    nodes: List[ListNode] = []
    curr = head
    while curr is not None:
        nodes.append(curr)
        curr = curr.next

    for start in range(0, len(nodes) - len(nodes) % k, k):
        left, right = start, start + k - 1
        while left < right:
            nodes[left], nodes[right] = nodes[right], nodes[left]
            left += 1
            right -= 1

    for i in range(len(nodes) - 1):
        nodes[i].next = nodes[i + 1]
    if nodes:
        nodes[-1].next = None
    return nodes[0] if nodes else None

### Complexity

- Time: $O(n)$
- Space: $O(n)$ for the node array

## Optimized insight

Use in-place reversal plus four named pointers. For a segment

```
... → A → 1 → 2 → 3 → B → ...
```

with $k = 3$:

```
      groupPrev
          ↓
... → A → 1 → 2 → 3 → B → ...
                    ↑    ↑
                   kth  groupNext
```

After reversing that group:

```
... → A → 3 → 2 → 1 → B → ...
          ↑         ↑
        new        new
       start       tail
```

`kth` becomes the new head of the group. The old first node becomes
the new tail, and therefore the `groupPrev` of the *next* group.

## Why a dummy node

Reversing the first group changes the real head (`1 → 2` becomes
`2 → 1`). If dummy sits in front:

```
dummy → 1 → 2 → 3 → 4
```

then every group has a predecessor, including the first. Return
`dummy.next` at the end.

## Optimized approach

1. `dummy.next = head`, `groupPrev = dummy`.
2. Repeat:
   - Walk $k$ steps from `groupPrev` to find `kth`.
   - If `kth` is missing, leftover suffix is shorter than $k$ — stop.
   - `groupNext = kth.next`.
   - Reverse the open interval from `groupPrev.next` up to (not
     including) `groupNext`, starting with `prev = groupNext`.
   - `oldStart = groupPrev.next` (this is the new tail).
   - `groupPrev.next = kth` (this is the new head of the group).
   - `groupPrev = oldStart`.
3. Return `dummy.next`.

### The four pointers

| Pointer | Job |
|---|---|
| `groupPrev` | node immediately before the current group |
| `kth` | last node of the current group |
| `groupNext` | first node after the current group |
| `curr` | node currently being reversed |

### Step-by-step trace — $k = 2$

Start:

```
dummy → 1 → 2 → 3 → 4 → 5
  ↑
groupPrev
```

First group: `kth = 2`, `groupNext = 3`. Reverse with `prev = 3`,
`curr = 1`.

| Step | temp | rewired | list after the write |
|---|---|---|---|
| 1 | 2 | `1.next = 3` | `1 → 3 → 4 → 5`, `curr` moves to 2 |
| 2 | 3 | `2.next = 1` | `2 → 1 → 3 → 4 → 5`, `curr` hits `groupNext` |

Reconnect dummy to `kth`:

```
dummy → 2 → 1 → 3 → 4 → 5
              ↑
          groupPrev (old start)
```

Second group: `kth = 4`, `groupNext = 5`. Reverse `3 → 4` into
`4 → 3`:

```
dummy → 2 → 1 → 4 → 3 → 5
```

Third group is only `5`. `getKth` returns `None`. Stop.

Result: $2 \rightarrow 1 \rightarrow 4 \rightarrow 3 \rightarrow 5$.

### Trace — $k = 3$

Complete group `[1 → 2 → 3]` reverses to `[3 → 2 → 1]`. Suffix
`4 → 5` has length $2 < k$, so it stays.

Result: $3 \rightarrow 2 \rightarrow 1 \rightarrow 4 \rightarrow 5$.

In [ ]:
class Solution:
    def reverseKGroup(self, head: Optional[ListNode], k: int) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        groupPrev = dummy

        while True:
            kth = self.getKth(groupPrev, k)
            if kth is None:
                break

            groupNext = kth.next

            # prev = groupNext (not None) so the new tail already
            # points at the unused suffix.
            prev = groupNext
            curr = groupPrev.next

            while curr is not groupNext:
                temp = curr.next
                curr.next = prev
                prev = curr
                curr = temp

            oldGroupStart = groupPrev.next
            groupPrev.next = kth
            groupPrev = oldGroupStart

        return dummy.next

    def getKth(self, curr: Optional[ListNode], k: int) -> Optional[ListNode]:
        while curr is not None and k > 0:
            curr = curr.next
            k -= 1
        return curr


def optimized_k_group(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    return Solution().reverseKGroup(head, k)

## Complexity analysis

Each node is visited a constant number of times: once while locating
its group, once while reversing it (if it belongs to a complete group).

- Time: $O(n)$
- Space: $O(1)$ extra — `dummy`, `groupPrev`, `kth`, `groupNext`,
  `prev`, `curr`, `temp`. No auxiliary list of size $n$.

That is the strongest form of the problem (in-place, constant extra
memory).

## Edge cases

- $k = 1$: every group is a single node; the list is unchanged
- $k = n$: the whole list reverses
- leftover suffix shorter than $k$: must stay in original order
- two nodes, $k = 2$: reduces to a single swap
- values are irrelevant; only `next` pointers move

## Testing

In [ ]:
def run_k_group(values: List[int], k: int) -> List[int]:
    return to_list(optimized_k_group(from_list(values), k))


assert run_k_group([1, 2, 3, 4, 5], 2) == [2, 1, 4, 3, 5]
assert run_k_group([1, 2, 3, 4, 5], 3) == [3, 2, 1, 4, 5]
assert run_k_group([1, 2, 3], 1) == [1, 2, 3]
assert run_k_group([1, 2, 3, 4], 4) == [4, 3, 2, 1]
assert run_k_group([1], 1) == [1]
assert to_list(brute_force_k_group(from_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_list(brute_force_k_group(from_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]

print("Reverse Nodes in k-Group checks passed")

## Alternative approaches

- Array of nodes, reverse chunks, rewire: $O(n)$ time, $O(n)$ space.
  Useful as a check, not the intended solution.
- Recursion: reverse the first $k$ nodes, then recurse on the rest.
  Same $O(n)$ time, but $O(n / k)$ stack space.
- Swap values in each group: forbidden by the problem statement.

## Mistakes I made

- Reversing the leftover suffix when $n$ is not a multiple of $k$.
- Starting the reversal with `prev = None`, which detaches the group
  from the rest of the list.
- Losing `groupPrev.next` before reconnecting — save `oldGroupStart`
  first.
- Off-by-one in `getKth`: it must move exactly $k$ times from the node
  *before* the group.

## Pattern recognition

Standard linked-list reversal:

```text
save next → reverse pointer → advance prev → advance curr
```

with one change:

```python
prev = groupNext   # not None
```

That single assignment is what stitches each reversed block back onto
the unused suffix.

## Related problems

- Reverse Linked List — the inner loop with `prev = None`
- Swap Nodes in Pairs — this problem with $k = 2$
- Reverse Linked List II — reverse one closed index range
- Rotate List — regroup pointers, no reversal of values' order in
  the same way

## Real-world or engineering connection

Chunked in-place reversal shows up any time a singly linked buffer
must be rewritten in fixed-size blocks (protocol frames, disk-page
lists) without allocating a second copy of the chain.

## Final takeaways

Do not memorize the whole function. Keep this loop:

```python
prev = groupNext
curr = groupPrev.next

while curr is not groupNext:
    temp = curr.next
    curr.next = prev
    prev = curr
    curr = temp
```

`kth` becomes the new head of the group. The original first node
becomes the new tail and the predecessor of the next group.

## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-09-01 | 1 | First write-up: dummy + getKth + reverse onto groupNext |

# Koko Eating Bananas

## Metadata

- Source: NeetCode / LeetCode 875
- Problem URL: https://leetcode.com/problems/koko-eating-bananas/
- Difficulty: Medium
- Topic: Binary Search (on the answer)
- Date started: 2026-09-06
- Date solved: 2026-09-06
- Current mastery level: 1
- Last reviewed: 2026-09-06
- Next review:


## Problem statement in my own words

Koko has `n` piles of bananas. Pile `i` has `piles[i]` bananas. She has
exactly `h` hours to finish every pile.

She picks one integer eating speed `k` (bananas per hour) and keeps that
speed for the whole job. Each hour she chooses **one** pile and eats
`k` bananas from it:

- if the pile has at least `k` bananas, she eats `k` and that hour is done
- if the pile has fewer than `k`, she finishes the pile and **cannot**
  start another pile in the same hour

So a pile of `p` bananas always takes $\lceil p / k \rceil$ hours.

Return the **smallest** integer `k` such that the total hours needed is
at most `h`.


## Inputs, outputs, and constraints

- Inputs: `piles` (list of positive ints) and `h` (int hours)
- Outputs: `int` — the minimum feasible eating speed `k`
- Constraints:
  - $1 \le n \le 10^4$ where $n = $ `len(piles)`
  - $n \le h \le 10^9$
  - $1 \le$ `piles[i]` $\le 10^9$

The $h \ge n$ guarantee matters: eating at $k = \max(\text{piles})$
always finishes in exactly $n$ hours (one pile per hour), so a feasible
speed always exists.


## Examples

| Input | Expected | Notes |
|---|---|---|
| `piles = [1, 4, 3, 2]`, `h = 9` | `2` | $T(2)=6 \le 9$, but $T(1)=10 > 9$ |
| `piles = [25, 10, 23, 4]`, `h = 4` | `25` | $h = n$, so every pile must take 1 hour |

Hours for one pile: $\lceil p / k \rceil = (p + k - 1) // k$ in integer
arithmetic.


## Initial observations

- We are **not** searching the piles. We are searching the possible
  speeds $k$.
- Define $T(k) = \sum_i \lceil p_i / k \rceil$. We want the smallest
  $k$ with $T(k) \le h$.
- $T(k)$ is monotonic: larger $k$ can only reduce (or hold) the hours.
  So the feasibility array looks like `F F F T T T ...` and we want the
  first `T`.
- That is binary search on the answer (lower bound / parametric search).
- Search range: $k \in [1, \max(piles)]$. Speed $1$ is the slowest
  legal rate. Speed $\max(piles)$ is always enough because $h \ge n$.
- Brute-forcing every $k$ up to $10^9$ is impossible. Binary search
  cuts that to about $\log_2(10^9) \approx 30$ candidate speeds.


## Brute-force approach

### Why it works

Try $k = 1, 2, 3, \ldots$ and return the first speed whose total hours
are $\le h$. Correct because of monotonicity. Too slow when
$\max(piles)$ is near $10^9$.

### Implementation


In [ ]:
from typing import List


def hours_needed(piles: List[int], k: int) -> int:
    """Hours Koko needs if she eats at speed k."""
    total = 0
    for pile in piles:
        total += (pile + k - 1) // k
    return total


def brute_force(piles: List[int], h: int) -> int:
    for k in range(1, max(piles) + 1):
        if hours_needed(piles, k) <= h:
            return k
    return max(piles)


### Complexity

- Time: $O(n \cdot M)$ where $M = \max(piles)$ — up to about
  $10^4 \times 10^9$ pile checks. Not viable.
- Space: $O(1)$


## Optimized insight

Feasibility is a monotonic predicate on an integer range, so binary
search the speeds instead of scanning them.

$$
k_1 < k_2 \quad\Longrightarrow\quad T(k_1) \ge T(k_2)
$$

Conceptually:

```
k:  1  2  3  4  5  6  7  8 ...
    F  F  F  F  T  T  T  T ...
                ^
             first valid  ← this is the answer
```

Ceiling division without floats:

$$
\left\lceil \frac{p}{k} \right\rceil = (p + k - 1) // k
$$

Example: $p = 3$, $k = 2$ → $(3 + 2 - 1) // 2 = 2$.

## Optimized approach

1. `left = 1`, `right = max(piles)`
2. While `left < right`:
   - `mid = left + (right - left) // 2`
   - compute $T(\text{mid})$
   - if $T(\text{mid}) \le h$: `mid` works, but a smaller speed might
     also work → `right = mid` (keep `mid`)
   - else: `mid` is too slow → `left = mid + 1`
3. Return `left` (`left == right` is the first feasible speed)

Keep `mid` on success. Discard it on failure.

### Step-by-step trace — Example 1

`piles = [1, 4, 3, 2]`, `h = 9`. Range starts as `[1, 4]`.

| Step | left | right | mid | $T(\text{mid})$ | Note |
|---|---|---|---|---|---|
| 1 | 1 | 4 | 2 | $1+2+2+1 = 6 \le 9$ | works → `right = 2` |
| 2 | 1 | 2 | 1 | $1+4+3+2 = 10 > 9$ | too slow → `left = 2` |

`left == right == 2`. Answer: $k = 2$.

### Trace — Example 2

`piles = [25, 10, 23, 4]`, `h = 4`. Four piles and four hours, so every
pile must finish in one hour. That forces $k \ge 25$.

$T(25) = 1+1+1+1 = 4 \le 4$. Answer: $k = 25$.


In [ ]:
from typing import List


class Solution:
    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        # Slowest legal speed.
        left = 1

        # This fast, every pile takes at most one hour — always enough.
        right = max(piles)

        # Smallest feasible speed in [left, right].
        while left < right:
            mid = left + (right - left) // 2

            needed = 0
            for pile in piles:
                needed += (pile + mid - 1) // mid

            if needed <= h:
                # mid works; a slower speed might also work.
                right = mid
            else:
                # mid is too slow; drop it and everything below it.
                left = mid + 1

        return left


def optimized(piles: List[int], h: int) -> int:
    return Solution().minEatingSpeed(piles, h)


## Complexity analysis

Let $n = $ `len(piles)` and $M = \max(piles)$.

Each candidate speed inspects every pile: $O(n)$.

Binary search tests $O(\log M)$ speeds.

- Time: $O(n \log M)$
- Space: $O(1)$

Worst case is about $10^4 \times 30 = 3 \times 10^5$ pile checks —
fine. Linear search over $k$ would be up to $10^9$ candidates.


## Edge cases

- $h = n$: answer is exactly `max(piles)` (one hour per pile)
- one pile: answer is $\lceil p / h \rceil$
- every pile is size $1$: answer is $1$
- a single huge pile plus tiny piles: the large pile dominates $T(k)$
- $k$ in the middle of a plateau of $T(k)$ values — still return the
  *leftmost* feasible $k$, which the `right = mid` policy finds


## Testing


In [ ]:
assert optimized([1, 4, 3, 2], 9) == 2
assert optimized([25, 10, 23, 4], 4) == 25
assert optimized([10], 5) == 2
assert optimized([1, 1, 1], 3) == 1
assert optimized([30, 11, 23, 4, 20], 6) == 23
assert brute_force([1, 4, 3, 2], 9) == 2
assert brute_force([25, 10, 23, 4], 4) == 25
assert brute_force([10], 5) == 2

print("Koko Eating Bananas checks passed")


## Visualization

The plot below steps through the **same loop as the Python code**.
Each frame shows:

- the current `[left, mid, right]` window on the speed number line
- whether that `mid` is too slow (`F`) or feasible (`T`)
- hours per pile at the candidate speed, and $T(k)$ vs $h$

If `ipywidgets` is installed, use **Next step** / **Prev** / **Reset**.
Otherwise every iteration is drawn as a static row of frames.


In [ ]:
from dataclasses import dataclass
from typing import List, Optional

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch


@dataclass(frozen=True)
class KokoFrame:
    step: int
    left: int
    right: int
    mid: Optional[int]
    hours: Optional[int]
    decision: str


def koko_search_frames(piles: List[int], h: int) -> List[KokoFrame]:
    """Record left / mid / right exactly as minEatingSpeed executes."""
    frames: List[KokoFrame] = []
    left = 1
    right = max(piles)
    step = 1

    while left < right:
        mid = left + (right - left) // 2
        hours = hours_needed(piles, mid)
        if hours <= h:
            frames.append(KokoFrame(step, left, right, mid, hours, "works → right = mid"))
            right = mid
        else:
            frames.append(KokoFrame(step, left, right, mid, hours, "too slow → left = mid + 1"))
            left = mid + 1
        step += 1

    frames.append(KokoFrame(step, left, right, None, hours_needed(piles, left), "done"))
    return frames


def _speed_colors(max_k: int, h: int, piles: List[int]) -> List[str]:
    colors = []
    for k in range(1, max_k + 1):
        colors.append("#2a9d8f" if hours_needed(piles, k) <= h else "#e76f51")
    return colors


def plot_koko_frame(
    piles: List[int],
    h: int,
    frame: KokoFrame,
    *,
    ax_piles=None,
    ax_speeds=None,
):
    """Two-panel snapshot of one binary-search iteration."""
    max_k = max(piles)
    created_fig = ax_piles is None
    if created_fig:
        _, (ax_piles, ax_speeds) = plt.subplots(
            1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.1, 1.4]}
        )

    k_view = frame.mid if frame.mid is not None else frame.left
    pile_hours = [(pile + k_view - 1) // k_view for pile in piles]
    bars = ax_piles.bar(
        range(len(piles)),
        piles,
        color="#4c78a8",
        edgecolor="black",
        linewidth=0.6,
    )
    for idx, (bar, pile, took) in enumerate(zip(bars, piles, pile_hours)):
        ax_piles.text(
            bar.get_x() + bar.get_width() / 2,
            pile + max(piles) * 0.03,
            f"{took}h",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax_piles.set_xticks(range(len(piles)))
    ax_piles.set_xticklabels([f"pile {i}\n{p}" for i, p in enumerate(piles)])
    ax_piles.set_ylabel("bananas")
    ax_piles.set_ylim(0, max(piles) * 1.25)
    ax_piles.set_title(f"Hours per pile at k = {k_view}")

    xs = list(range(1, max_k + 1))
    colors = _speed_colors(max_k, h, piles)
    ax_speeds.bar(xs, [1] * max_k, color=colors, edgecolor="white", width=0.8)
    ax_speeds.set_yticks([])
    ax_speeds.set_xticks(xs)
    ax_speeds.set_xlabel("eating speed k")
    ax_speeds.set_xlim(0.4, max_k + 0.6)
    ax_speeds.set_ylim(-0.55, 1.85)

    def _mark(x: int, label: str, y: float, color: str) -> None:
        ax_speeds.annotate(
            label,
            xy=(x, 1.02),
            xytext=(x, y),
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
            color=color,
            arrowprops={"arrowstyle": "->", "color": color, "lw": 1.2},
        )

    _mark(frame.left, "L", 1.45, "#1d3557")
    _mark(frame.right, "R", 1.45, "#1d3557")
    if frame.mid is not None:
        _mark(frame.mid, "M", 1.68, "#9b2226")
        ax_speeds.bar(
            [frame.mid],
            [1],
            color="none",
            edgecolor="#9b2226",
            linewidth=2.4,
            width=0.8,
        )

    hours_txt = "?" if frame.hours is None else str(frame.hours)
    title = (
        f"Step {frame.step}   L={frame.left}  "
        f"M={frame.mid if frame.mid is not None else '—'}  "
        f"R={frame.right}   T(k)={hours_txt}  h={h}\n{frame.decision}"
    )
    ax_speeds.set_title(title, loc="left")

    legend_y = -0.42
    ax_speeds.add_patch(
        FancyBboxPatch((0.55, legend_y - 0.08), 0.22, 0.16, boxstyle="round,pad=0.02",
                       facecolor="#e76f51", edgecolor="none")
    )
    ax_speeds.text(0.85, legend_y, "T(k) > h  (too slow)", va="center", fontsize=8)
    ax_speeds.add_patch(
        FancyBboxPatch((max_k * 0.45, legend_y - 0.08), 0.22, 0.16, boxstyle="round,pad=0.02",
                       facecolor="#2a9d8f", edgecolor="none")
    )
    ax_speeds.text(max_k * 0.45 + 0.32, legend_y, "T(k) ≤ h  (feasible)", va="center", fontsize=8)

    if created_fig:
        plt.tight_layout()
    return ax_piles, ax_speeds


def visualize_koko_all(piles: List[int], h: int) -> None:
    """Draw every binary-search iteration as its own figure."""
    frames = koko_search_frames(piles, h)
    print(f"piles={piles}  h={h}  answer={frames[-1].left}")
    print(f"{'step':>4}  {'left':>4}  {'mid':>4}  {'right':>5}  {'T(k)':>5}  decision")
    for frame in frames:
        mid = "—" if frame.mid is None else frame.mid
        print(
            f"{frame.step:>4}  {frame.left:>4}  {mid:>4}  {frame.right:>5}  "
            f"{frame.hours:>5}  {frame.decision}"
        )
        plot_koko_frame(piles, h, frame)
    plt.show()


def step_through_koko(piles: List[int] = [1, 4, 3, 2], h: int = 9) -> None:
    """Interactive Next-step walker; falls back to all frames."""
    frames = koko_search_frames(piles, h)
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output
    except ImportError:
        visualize_koko_all(piles, h)
        return

    output = widgets.Output()
    state = {"i": 0}

    def render() -> None:
        with output:
            clear_output(wait=True)
            frame = frames[state["i"]]
            print(
                f"Example piles={piles}  h={h}   "
                f"frame {state['i'] + 1}/{len(frames)}"
            )
            plot_koko_frame(piles, h, frame)
            plt.show()

    def on_next(_=None) -> None:
        state["i"] = min(state["i"] + 1, len(frames) - 1)
        render()

    def on_prev(_=None) -> None:
        state["i"] = max(state["i"] - 1, 0)
        render()

    def on_reset(_=None) -> None:
        state["i"] = 0
        render()

    prev_btn = widgets.Button(description="Prev")
    next_btn = widgets.Button(description="Next step")
    reset_btn = widgets.Button(description="Reset")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    reset_btn.on_click(on_reset)

    display(widgets.HBox([prev_btn, next_btn, reset_btn]), output)
    render()


print("Example 1 — piles = [1, 4, 3, 2], h = 9")
step_through_koko([1, 4, 3, 2], 9)

print("\nExample 2 — piles = [25, 10, 23, 4], h = 4")
step_through_koko([25, 10, 23, 4], 4)


## Alternative approaches

- Linear scan of $k = 1 \ldots M$: correct, $O(nM)$, fails the limits.
- Binary search with `left <= right` and `ans = mid`: same result if the
  update rules stay consistent. The `while left < right` / `right = mid`
  form is the lower-bound template.
- `math.ceil(pile / k)`: same math, but uses floats. Prefer
  `(pile + k - 1) // k`.

## Mistakes I made

- Searching the piles instead of the speeds.
- Using `right = mid - 1` when `mid` works — that can drop the answer.
- Exclusive upper bound `right = max(piles) + 1` mixed with a
  `left <= right` loop. Pick one convention.
- Forgetting $h \ge n$, which is why `right = max(piles)` is safe.

## Pattern recognition

Ask: *is there a range of answers, and once one value works, every
larger value also works?* If yes, binary-search the answer.

```python
while left < right:
    mid = left + (right - left) // 2
    if works(mid):
        right = mid
    else:
        left = mid + 1
return left
```

Same skeleton for minimum capacity, minimum rate, minimum time, or any
first-feasible threshold.

## Related problems

- Search a 2D Matrix — binary search on **indices**, not on the answer
- Capacity To Ship Packages Within D Days — same $T(k)$ idea, ships
  instead of bananas
- Split Array Largest Sum — minimize a threshold under a count budget
- Minimum Number of Days to Make m Bouquets — monotonic feasibility
  on a date
- Find First and Last Position — lower-bound on a sorted array

## Real-world or engineering connection

This is rate / capacity sizing: the smallest server throughput, disk
bandwidth, or infusion rate that still meets a deadline when work must
be processed in indivisible chunks. The math is the same: a monotonic
cost function over an integer resource, solved by binary search.

## Final takeaways

Do not memorize the banana story. Keep this:

1. $T(k) = \sum \lceil p_i / k \rceil$
2. $T$ decreases as $k$ increases
3. search $k \in [1, \max(piles)]$
4. `works → right = mid`, `fails → left = mid + 1`

The reusable pattern is binary search on a monotonic predicate, not
binary search on a sorted array.


## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-09-06 | 1 | First write-up: binary search on eating speed $k$ |


# Find Minimum in Rotated Sorted Array

## Metadata

- Source: NeetCode / LeetCode 153
- Problem URL: https://leetcode.com/problems/find-minimum-in-rotated-sorted-array/
- Difficulty: Medium
- Topic: Binary Search
- Date started: 2026-09-14
- Date solved: 2026-09-14
- Current mastery level: 1
- Last reviewed: 2026-09-14
- Next review:


## Problem statement in my own words

Start with a strictly increasing array of unique integers. Then rotate it
between $1$ and $n$ times: each rotation peels the last element off the
end and moves it to the front.

```
original:          [1, 2, 3, 4, 5, 6]
rotated 4 times:   [3, 4, 5, 6, 1, 2]
rotated n times:   [1, 2, 3, 4, 5, 6]   ← back to sorted
```

The result is still two sorted runs glued together. The minimum is the
**rotation point** — the start of the original array, now sitting
somewhere in the middle (or at index $0$ if the array landed fully
sorted).

Return that minimum. A linear scan is $O(n)$ and always correct. The
asked bound is $O(\log n)$.


## Inputs, outputs, and constraints

- Inputs: `nums` — a rotated sorted array of unique integers
- Outputs: `int` — the smallest value in `nums`
- Constraints:
  - $1 \le n \le 1000$ where $n = $ `len(nums)`
  - $-1000 \le$ `nums[i]` $\le 1000$
  - every value is unique
  - `nums` is a rotation of a strictly increasing array


## Examples

| Input | Expected | Notes |
|---|---|---|
| `nums = [3, 4, 5, 6, 1, 2]` | `1` | rotation point at index $4$ |
| `nums = [4, 5, 0, 1, 2, 3]` | `0` | rotation point at index $2$ |
| `nums = [4, 5, 6, 7]` | `4` | rotated $n$ times; already sorted |

The last example is the trap: “rotated” includes the identity rotation,
so the minimum can be `nums[0]`.


## Initial observations

- Unique values mean there is exactly one drop: some index $i$ with
  `nums[i] < nums[i - 1]`. That `nums[i]` is the minimum. If there is
  no drop, the array is fully sorted and the minimum is `nums[0]`.
- The drop splits the array into two increasing runs. Every value in
  the left run is **larger** than every value in the right run.
- `nums[right]` is a witness. Compare `nums[mid]` against it:
  - `nums[mid] > nums[right]` → `mid` sits in the large left run, so
    the minimum is strictly to the right of `mid`
  - `nums[mid] < nums[right]` → `mid` sits in the small right run
    (or the array is unrotated), so the minimum is at `mid` or to its
    left
- Because values are unique, `nums[mid] == nums[right]` cannot happen
  while `mid < right`. The loop invariant `left < right` plus
  `mid = left + (right - left) // 2` keeps `mid` strictly less than
  `right`.
- This is the same `while left < right` / `right = mid` skeleton as
  Koko: keep a closed window that is guaranteed to contain the answer,
  and shrink it until one index remains.


## Brute-force approach

### Why it works

Walk the array once and keep the smallest value seen. Rotation is
ignored. Correct for any array, not just a rotated sorted one.

### Implementation


In [ ]:
from typing import List


def brute_force(nums: List[int]) -> int:
    minimum = nums[0]
    for value in nums:
        if value < minimum:
            minimum = value
    return minimum


### Complexity

- Time: $O(n)$
- Space: $O(1)$

Fine for $n \le 1000$. The problem asks for $O(\log n)$ anyway, which
is the version you want in interviews.


## Optimized insight

The search space is the **index range**, not a reconstructed unrotated
copy. The predicate is:

> is `mid` still in the large left run?

```
index:     0  1  2  3  4  5
nums:     [3, 4, 5, 6, 1, 2]
           \____ left ____/  \ right /
                 all > 2         min lives here

mid=2 → 5 > nums[5]=2  → drop [0, 2], keep [3, 5]
mid=4 → 1 < nums[5]=2  → drop (4, 5], keep [3, 4]
mid=3 → 6 > nums[4]=1  → drop [3, 3], keep [4, 4]
answer = nums[4] = 1
```

If `mid` is larger than the rightmost remaining value, everything from
`left` through `mid` is in the large run and can go. Otherwise `mid`
itself might be the minimum, so it stays.

That is why success uses `right = mid` (keep `mid`) and failure uses
`left = mid + 1` (discard `mid`). Same lower-bound template as Koko,
different predicate.

## Optimized approach

1. `left = 0`, `right = n - 1`
2. While `left < right`:
   - `mid = left + (right - left) // 2`
   - if `nums[mid] > nums[right]`: `left = mid + 1`
   - else: `right = mid`
3. Return `nums[left]` (`left == right`)

### Step-by-step trace — Example 1

`nums = [3, 4, 5, 6, 1, 2]`. Window starts as `[0, 5]`.

| Step | left | mid | right | `nums[mid]` | `nums[right]` | Note |
|---|---|---|---|---|---|---|
| 1 | 0 | 2 | 5 | 5 | 2 | $5 > 2$ → `left = 3` |
| 2 | 3 | 4 | 5 | 1 | 2 | $1 < 2$ → `right = 4` |
| 3 | 3 | 3 | 4 | 6 | 1 | $6 > 1$ → `left = 4` |

`left == right == 4`. Answer: `1`.

### Trace — Example 2

`nums = [4, 5, 0, 1, 2, 3]`.

| Step | left | mid | right | `nums[mid]` | `nums[right]` | Note |
|---|---|---|---|---|---|---|
| 1 | 0 | 2 | 5 | 0 | 3 | $0 < 3$ → `right = 2` |
| 2 | 0 | 1 | 2 | 5 | 0 | $5 > 0$ → `left = 2` |

`left == right == 2`. Answer: `0`.

### Trace — Example 3 (already sorted)

`nums = [4, 5, 6, 7]`.

| Step | left | mid | right | `nums[mid]` | `nums[right]` | Note |
|---|---|---|---|---|---|---|
| 1 | 0 | 1 | 3 | 5 | 7 | $5 < 7$ → `right = 1` |
| 2 | 0 | 0 | 1 | 4 | 5 | $4 < 5$ → `right = 0` |

`left == right == 0`. Answer: `4`. The `>` branch never fires, so the
window only shrinks from the right and lands on index $0$.


In [ ]:
from typing import List


class Solution:
    def findMin(self, nums: List[int]) -> int:
        # Search interval that is guaranteed to contain the minimum.
        left = 0
        right = len(nums) - 1

        # Continue until only one candidate remains.
        while left < right:
            mid = left + (right - left) // 2

            # If mid is larger than the rightmost value,
            # the rotation point/minimum must be to the right.
            if nums[mid] > nums[right]:
                left = mid + 1

            # Otherwise, mid could itself be the minimum,
            # so keep mid and eliminate everything to its right.
            else:
                right = mid

        # left == right, so this index must contain the minimum.
        return nums[left]


def optimized(nums: List[int]) -> int:
    return Solution().findMin(nums)


## Complexity analysis

Each iteration discards at least half of the remaining indices
(`mid` itself, plus one side).

$$

n \rightarrow n/2 \rightarrow n/4 \rightarrow \cdots \rightarrow 1
$$


- Time: $O(\log n)$
- Space: $O(1)$ — only `left`, `right`, `mid`


## Edge cases

- $n = 1$: `left == right` immediately; return `nums[0]`
- already sorted (rotated $n$ times): always take the `right = mid`
  branch and converge on index $0$
- rotated by $1$: minimum is `nums[-1]`; the `>` branch walks `left`
  all the way to $n - 1$
- two elements `[2, 1]`: one comparison, `left` becomes $1$
- two elements `[1, 2]`: one comparison, `right` becomes $0$
- negatives are legal; only order matters
- uniqueness is load-bearing — duplicates would make
  `nums[mid] == nums[right]` possible and this loop would be wrong
  (that is LeetCode 154)


## Testing


In [ ]:
assert optimized([3, 4, 5, 6, 1, 2]) == 1
assert optimized([4, 5, 0, 1, 2, 3]) == 0
assert optimized([4, 5, 6, 7]) == 4
assert optimized([1]) == 1
assert optimized([2, 1]) == 1
assert optimized([1, 2]) == 1
assert optimized([-1, 0, 3, -2]) == -2
assert brute_force([3, 4, 5, 6, 1, 2]) == 1
assert brute_force([4, 5, 6, 7]) == 4
assert brute_force([1]) == 1

print("Find Minimum in Rotated Sorted Array checks passed")


## Visualization

Two views of the same idea:

1. **Rotation shape** — a sorted array is a rising ramp. After a
   rotation it becomes two ramps, and the minimum is the valley where
   they meet.
2. **Binary-search walk** — each frame is one iteration of `findMin`.
   Bars in the live window stay colored; discarded indices fade.
   `L` / `M` / `R` mark the current pointers. The decision is the
   comparison `nums[mid] ? nums[right]`.

If `ipywidgets` is installed, use **Next step** / **Prev** / **Reset**.
Otherwise every iteration is drawn as a static row of frames.


In [ ]:
from dataclasses import dataclass
from typing import List, Optional

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch


@dataclass(frozen=True)
class FindMinFrame:
    step: int
    left: int
    right: int
    mid: Optional[int]
    decision: str


def find_min_frames(nums: List[int]) -> List[FindMinFrame]:
    """Record left / mid / right exactly as findMin executes."""
    frames: List[FindMinFrame] = []
    left = 0
    right = len(nums) - 1
    step = 1

    while left < right:
        mid = left + (right - left) // 2
        if nums[mid] > nums[right]:
            frames.append(
                FindMinFrame(
                    step,
                    left,
                    right,
                    mid,
                    f"nums[mid]={nums[mid]} > nums[right]={nums[right]}  →  left = mid + 1",
                )
            )
            left = mid + 1
        else:
            frames.append(
                FindMinFrame(
                    step,
                    left,
                    right,
                    mid,
                    f"nums[mid]={nums[mid]} ≤ nums[right]={nums[right]}  →  right = mid",
                )
            )
            right = mid
        step += 1

    frames.append(
        FindMinFrame(step, left, right, None, f"done  →  min = nums[{left}] = {nums[left]}")
    )
    return frames


def plot_rotation_shape(original: List[int], rotated: List[int]) -> None:
    """Show a sorted ramp versus the two-ramp valley after rotation."""
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6), sharey=True)
    for ax, values, title in (
        (axes[0], original, "sorted (before rotation)"),
        (axes[1], rotated, "rotated — min is the valley"),
    ):
        xs = list(range(len(values)))
        colors = ["#4c78a8"] * len(values)
        if values:
            colors[values.index(min(values))] = "#2a9d8f"
        ax.bar(xs, values, color=colors, edgecolor="black", linewidth=0.6)
        ax.plot(xs, values, color="#1d3557", marker="o", linewidth=1.4)
        ax.set_xticks(xs)
        ax.set_xlabel("index")
        ax.set_title(title)
        ax.set_ylabel("value")
        min_i = values.index(min(values))
        ax.annotate(
            f"min = {values[min_i]}",
            xy=(min_i, values[min_i]),
            xytext=(min_i, max(values) * 0.55 if max(values) else 1),
            ha="center",
            fontsize=9,
            color="#1b6b62",
            arrowprops={"arrowstyle": "->", "color": "#1b6b62", "lw": 1.1},
        )
    fig.suptitle("Rotation splits one increasing run into two", fontsize=12)
    plt.tight_layout()
    plt.show()


def plot_find_min_frame(nums: List[int], frame: FindMinFrame, *, ax=None):
    """One binary-search iteration on the rotated array."""
    created_fig = ax is None
    if created_fig:
        _, ax = plt.subplots(figsize=(max(8.5, len(nums) * 1.15), 4.4))

    n = len(nums)
    xs = list(range(n))
    colors = []
    edges = []
    linewidths = []
    for i in xs:
        if i < frame.left or i > frame.right:
            colors.append("#d9d9d9")
            edges.append("#9a9a9a")
            linewidths.append(0.6)
        elif frame.mid is not None and i == frame.mid:
            colors.append("#e76f51")
            edges.append("#9b2226")
            linewidths.append(2.2)
        elif frame.mid is None and i == frame.left:
            colors.append("#2a9d8f")
            edges.append("#1b6b62")
            linewidths.append(2.2)
        else:
            colors.append("#4c78a8")
            edges.append("#1d3557")
            linewidths.append(0.8)

    bars = ax.bar(xs, nums, color=colors, edgecolor=edges, linewidth=linewidths)
    ax.plot(xs, nums, color="#1d3557", alpha=0.35, marker="o", linewidth=1.0)
    ax.set_xticks(xs)
    ax.set_xlabel("index")
    ax.set_ylabel("value")
    y_max = max(nums) if nums else 1
    y_min = min(nums) if nums else 0
    pad = max(1.0, (y_max - y_min) * 0.22)
    ax.set_ylim(y_min - pad * 0.35, y_max + pad)

    def _mark(idx: int, label: str, y_frac: float, color: str) -> None:
        ax.annotate(
            label,
            xy=(idx, nums[idx]),
            xytext=(idx, y_max + pad * y_frac),
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
            color=color,
            arrowprops={"arrowstyle": "->", "color": color, "lw": 1.2},
        )

    _mark(frame.left, "L", 0.55, "#1d3557")
    _mark(frame.right, "R", 0.55, "#1d3557")
    if frame.mid is not None:
        _mark(frame.mid, "M", 0.82, "#9b2226")

    for bar, value in zip(bars, nums):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (y_max - y_min) * 0.03,
            str(value),
            ha="center",
            va="bottom",
            fontsize=9,
        )

    mid_txt = "—" if frame.mid is None else str(frame.mid)
    ax.set_title(
        f"Step {frame.step}   L={frame.left}  M={mid_txt}  R={frame.right}\n{frame.decision}",
        loc="left",
    )

    legend_items = [
        (0.02, "#d9d9d9", "discarded"),
        (0.22, "#4c78a8", "still in window"),
        (0.48, "#e76f51", "mid"),
        (0.62, "#2a9d8f", "minimum"),
    ]
    for x, color, label in legend_items:
        ax.add_patch(
            FancyBboxPatch(
                (x, -0.18),
                0.03,
                0.08,
                boxstyle="round,pad=0.01",
                facecolor=color,
                edgecolor="black",
                linewidth=0.4,
                transform=ax.transAxes,
                clip_on=False,
            )
        )
        ax.text(x + 0.04, -0.14, label, transform=ax.transAxes, fontsize=8, va="center")

    if created_fig:
        plt.tight_layout()
    return ax


def visualize_find_min_all(nums: List[int]) -> None:
    """Draw every binary-search iteration as its own figure."""
    frames = find_min_frames(nums)
    print(f"nums={nums}  answer={nums[frames[-1].left]}")
    print(f"{'step':>4}  {'left':>4}  {'mid':>4}  {'right':>5}  decision")
    for frame in frames:
        mid = "—" if frame.mid is None else frame.mid
        print(
            f"{frame.step:>4}  {frame.left:>4}  {mid:>4}  {frame.right:>5}  {frame.decision}"
        )
        plot_find_min_frame(nums, frame)
    plt.show()


def step_through_find_min(nums: List[int]) -> None:
    """Interactive Next-step walker; falls back to all frames."""
    frames = find_min_frames(nums)
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output, display
    except ImportError:
        visualize_find_min_all(nums)
        return

    output = widgets.Output()
    state = {"i": 0}

    def render() -> None:
        with output:
            clear_output(wait=True)
            frame = frames[state["i"]]
            print(f"Example nums={nums}   frame {state['i'] + 1}/{len(frames)}")
            plot_find_min_frame(nums, frame)
            plt.show()

    def on_next(_=None) -> None:
        state["i"] = min(state["i"] + 1, len(frames) - 1)
        render()

    def on_prev(_=None) -> None:
        state["i"] = max(state["i"] - 1, 0)
        render()

    def on_reset(_=None) -> None:
        state["i"] = 0
        render()

    prev_btn = widgets.Button(description="Prev")
    next_btn = widgets.Button(description="Next step")
    reset_btn = widgets.Button(description="Reset")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    reset_btn.on_click(on_reset)

    display(widgets.HBox([prev_btn, next_btn, reset_btn]), output)
    render()


print("Rotation shape — [1, 2, 3, 4, 5, 6] rotated 4 times")
plot_rotation_shape([1, 2, 3, 4, 5, 6], [3, 4, 5, 6, 1, 2])

print("\nExample 1 — nums = [3, 4, 5, 6, 1, 2]")
step_through_find_min([3, 4, 5, 6, 1, 2])

print("\nExample 2 — nums = [4, 5, 0, 1, 2, 3]")
step_through_find_min([4, 5, 0, 1, 2, 3])

print("\nExample 3 — nums = [4, 5, 6, 7]  (already sorted)")
step_through_find_min([4, 5, 6, 7])


## Alternative approaches

- Linear `min(nums)`: $O(n)$, trivial, misses the point.
- Unrotate then read `nums[0]`: finding the drop is already $O(n)$.
- `while left <= right` with an explicit `ans` variable: same result if
  the updates stay consistent. The `left < right` / `right = mid` form
  is the lower-bound template.
- Recursive binary search: same $O(\log n)$ time, $O(\log n)$ stack.

## Mistakes I made

- Comparing `nums[mid]` to `nums[left]` instead of `nums[right]`. That
  breaks on an already-sorted array, where the left half is *not* the
  large run.
- Using `right = mid - 1` when `nums[mid] < nums[right]` — `mid` can
  be the minimum, so dropping it loses the answer.
- Inclusive `left <= right` mixed with `right = mid` → infinite loop
  when `left == mid`. Pick one convention.
- Assuming the array is never fully sorted. Rotating $n$ times is
  legal and the minimum is `nums[0]`.

## Pattern recognition

Closed window that must contain the answer; shrink with a predicate
on `mid` vs a known endpoint:

```python
while left < right:
    mid = left + (right - left) // 2
    if nums[mid] > nums[right]:
        left = mid + 1
    else:
        right = mid
return nums[left]
```

Same skeleton as Koko. Here the predicate is “is `mid` still in the
large left run?” instead of “does this speed finish in time?”

## Related problems

- Search in Rotated Sorted Array — same split, but hunt a target
  instead of the valley
- Find Minimum in Rotated Sorted Array II — duplicates; the
  `== nums[right]` case needs a linear shrink
- Koko Eating Bananas — identical `right = mid` lower-bound loop on
  a different predicate
- Search a 2D Matrix — binary search on a virtual index, no rotation
- Peak Index in a Mountain Array — another “find the turning point
  in $O(\log n)$” problem

## Real-world or engineering connection

Circular buffers and ring logs are rotated sorted sequences: the
logical start has slid, but each side of the wrap is still ordered.
Finding the wrap point in logarithmic probes is the same comparison
against the current right endpoint.

## Final takeaways

Do not unrotate the array. Keep this:

1. the minimum is the only drop
2. compare `nums[mid]` to `nums[right]`, not `nums[left]`
3. `>` means drop `mid` and the left; otherwise keep `mid`
4. stop when `left == right`

The reusable pattern is a shrinking window around a turning point,
not a search for a known target.


## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-09-14 | 1 | First write-up: compare mid to right, keep mid on the small side |
